# Cybersecurity Intrusion Detection with XAI and LLMs

Organized notebook for GitHub and academic reproducibility.

## Install Dependencies

In [ ]:
!pip install shap lime -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [ ]:
import os
# Standard library
import json

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Explainability
import shap
from lime import lime_tabular

# OpenAI API
from openai import OpenAI

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Model selection
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


## Configuration

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.30
SHAP_SAMPLE_SIZE = 200

DATASET_PATH = (
    "../datasets/dataset2/"
    "cybersecurity_intrusion_data.csv"
)


## Dataset Loading

In [ ]:
df = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", df.shape)

df.head()


Dataset Shape: (9537, 11)


,session_id,network_packet_size,protocol_type,login_attempts,session_duration,encryption_used,ip_reputation_score,failed_logins,browser_type,unusual_time_access,attack_detected
0,SID_00001,599,TCP,4,492.983263,DES,0.606818,1,Edge,0,1
1,SID_00002,472,TCP,3,1557.996461,DES,0.301569,0,Firefox,0,0
2,SID_00003,629,TCP,3,75.044262,DES,0.739164,2,Chrome,0,1
3,SID_00004,804,UDP,4,601.248835,DES,0.123267,0,Unknown,0,1
4,SID_00005,453,TCP,5,532.540888,AES,0.054874,1,Firefox,0,0


## Data Cleaning

In [ ]:
print(df.isnull().sum())

df.drop(columns=["session_id"], inplace=True)

df.drop_duplicates(inplace=True)
df.dropna(inplace=True)

print("Cleaned dataset shape:", df.shape)


session_id                0
network_packet_size       0
protocol_type             0
login_attempts            0
session_duration          0
encryption_used        1966
ip_reputation_score       0
failed_logins             0
browser_type              0
unusual_time_access       0
attack_detected           0
dtype: int64
Cleaned dataset shape: (7571, 10)


## Categorical Encoding

In [ ]:
categorical_columns = [
    "protocol_type",
    "encryption_used",
    "browser_type"
]

for column in categorical_columns:

    df[column] = (
        df[column]
        .astype("category")
    )

    print(
        f"{column} categories:",
        df[column].cat.categories.tolist()
    )

    df[column] = df[column].cat.codes


protocol_type categories: ['ICMP', 'TCP', 'UDP']
encryption_used categories: ['AES', 'DES']
browser_type categories: ['Chrome', 'Edge', 'Firefox', 'Safari', 'Unknown']


## Feature Scaling

In [ ]:
scaler = StandardScaler()

df[[
    "network_packet_size",
    "session_duration"
]] = scaler.fit_transform(
    df[[
        "network_packet_size",
        "session_duration"
    ]]
)


## Target Distribution

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    x="attack_detected",
    hue="attack_detected",
    data=df,
    palette="viridis",
    legend=False
)

plt.title(
    "Target Distribution "
    "(0: Normal | 1: Attack)"
)

plt.xlabel("attack_detected")
plt.ylabel("Count")

plt.show()


/tmp/ipykernel_1007549/1480408449.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Train/Test Split

In [ ]:
X = df.drop(columns=["attack_detected"])
y = df["attack_detected"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)


## Random Forest Training

In [ ]:
rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print(
    classification_report(
        y_test,
        y_pred
    )
)


Accuracy: 0.8948
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      1265
           1       1.00      0.77      0.87      1007

    accuracy                           0.89      2272
   macro avg       0.92      0.88      0.89      2272
weighted avg       0.91      0.89      0.89      2272



## Confusion Matrix

In [ ]:
matrix = confusion_matrix(
    y_test,
    y_pred
)

matrix = (
    matrix.astype("float")
    / matrix.sum(axis=1)[:, np.newaxis]
)

class_names = [
    "Normal",
    "Attack"
]

plt.figure(figsize=(6, 4))

sns.heatmap(
    matrix,
    annot=True,
    cmap=plt.cm.Blues,
    linewidths=0.2
)

plt.xlabel("Prediction")
plt.ylabel("Real")

plt.title("Dataset 2 Confusion Matrix")

plt.show()


/tmp/ipykernel_1007549/1102482136.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## SHAP Explainability

In [ ]:
sample_idx = np.random.choice(
    X_test.index,
    size=min(
        SHAP_SAMPLE_SIZE,
        len(X_test)
    ),
    replace=False
)

X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(
    rf_model
)

shap_values = explainer.shap_values(
    X_sample
)

shap.summary_plot(
    [
        shap_values[:, :, 0],
        shap_values[:, :, 1]
    ],
    X_sample,
    plot_type="bar",
    class_names=[
        "Normal",
        "Attack"
    ],
    show=False
)


## LIME Explainability

In [ ]:
feature_names = [
    "network_packet_size",
    "protocol_type",
    "login_attempts",
    "session_duration",
    "encryption_used",
    "ip_reputation_score",
    "failed_logins",
    "browser_type",
    "unusual_time_access"
]

class_names = [
    "Normal",
    "Attack"
]

lime_explainer = lime_tabular.LimeTabularExplainer(
    X_test.values,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    verbose=True
)

instance_index = 50

instance_to_explain = (
    X_test.iloc[instance_index]
    .values
)

lime_explanation = (
    lime_explainer.explain_instance(
        instance_to_explain,
        rf_model.predict_proba
    )
)

print(lime_explanation.as_list())


Intercept 0.6165134148446254
Prediction_local [0.23849023]
Right: 0.04
[('login_attempts <= 3.00', -0.15364284327236807), ('1.00 < failed_logins <= 2.00', -0.13714412016152053), ('ip_reputation_score <= 0.19', -0.08925745667338865), ('session_duration > 0.39', 0.03578600948140896), ('-0.71 < network_packet_size <= -0.02', 0.018497713838875107), ('protocol_type <= 1.00', -0.018134965167337678), ('0.00 < browser_type <= 2.00', -0.0167627458016047), ('encryption_used <= 0.00', -0.01114736044811602), ('unusual_time_access <= 0.00', -0.006217421202082248)]


/home/beloin/Documents/agents/Can-LLMs-Explain-AI-A-Study-of-XAI-in-Cybersecurity-ML-Models/venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


## LLM Context Information

In [ ]:
column_description = {

    "network_packet_size": (
        "Packet payload size"
    ),

    "protocol_type": (
        "Transport layer protocol"
    ),

    "login_attempts": (
        "Total login attempts"
    ),

    "session_duration": (
        "Session duration"
    ),

    "encryption_used": (
        "Encryption algorithm used"
    ),

    "ip_reputation_score": (
        "IP reputation score"
    ),

    "failed_logins": (
        "Number of failed logins"
    ),

    "browser_type": (
        "Browser type used"
    ),

    "unusual_time_access": (
        "Access at unusual time"
    ),

    "attack_detected": (
        "Normal or Attack"
    )
}

category_encoding = {

    "protocol_type": {
        "ICMP": 0,
        "TCP": 1,
        "UDP": 2
    },

    "encryption_used": {
        "AES": 0,
        "DES": 1
    },

    "browser_type": {
        "Chrome": 0,
        "Edge": 1,
        "Firefox": 2,
        "Safari": 3,
        "Unknown": 4
    }
}

train_sample = X_train.sample(50)

train_sample["attack_detected"] = (
    y_train.loc[train_sample.index]
)

train_sample_json = train_sample.to_json(
    orient="records"
)

pred_sample = pd.DataFrame({
    "real": y_test,
    "predicted": y_pred
})

pred_sample_json = pred_sample.sample(
    50
).to_json(
    orient="records"
)

model_info = {
    "model_type": "Random Forest",
    "task": (
        "Intrusion detection using logs"
    ),
    "target_variable": "attack_detected",
    "features": list(X.columns)
}


## Prompt Construction

In [ ]:
prompt = f"""
You are an expert in Explainable Artificial
Intelligence (XAI) and Cybersecurity.

=========================
MODEL INFORMATION
=========================
{model_info}

=========================
COLUMN DESCRIPTION
=========================
{column_description}

=========================
CATEGORY ENCODING
=========================
{category_encoding}

=========================
TRAINING DATA SAMPLE
=========================
{train_sample_json}

=========================
REAL vs PREDICTED SAMPLE
=========================
{pred_sample_json}

=========================
TASK
=========================

1. Identify the top-3 most relevant features.

2. Compare differences between classes.

3. Interpret model behavior in cybersecurity.

4. Avoid causal claims.

The explanation should be understandable
for technical and non-technical users.
"""


## OpenAI API Call

In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=300,
)

response = client.responses.create(
    model="gpt-5",
    input=[
        {
            "role": "system",
            "content": (
                "You are an expert in "
                "Machine Learning and XAI."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_output_tokens=12288
)


## Extract LLM Explanation

In [ ]:
explanation = ""

for item in response.output:

    if item.type == "message":

        for content in item.content:

            if content.type == "output_text":
                explanation += content.text

print(explanation)

Here’s a concise, mixed-audience explanation of what this Random Forest appears to be using to flag intrusions and how that aligns with security intuition. I avoid causal claims and describe model-learned associations.

1) Top-3 most relevant features (model-learned signals)
- ip_reputation_score (IP reputation score): Higher scores are frequently associated with the “attack_detected=1” class in the sample (often ≥ ~0.35–0.40). The forest likely splits on this early and often.
- failed_logins (Number of failed logins): Larger counts (e.g., ≥ 3) commonly appear in attack-labeled rows; small counts (0–1) are common in normal traffic.
- unusual_time_access (Access at unusual time): Values of 1 (unusual time) co-occur with attacks more than with normal traffic.

Close followers: login_attempts (higher counts linked with attacks), protocol_type (UDP/ICMP vs TCP), and session_duration (very short or very long sessions show up around attacks).

Note: This ranking is based on common RF importa

## Local LLM Integration

In [ ]:

# ============================================================
# Local LLM Integration
# ============================================================

# Adapt this section to test local/open-source LLMs
# such as:
#
# - GPT-OSS-20B
# - Llama 3
# - Mistral
# - DeepSeek
#
# Example frameworks:
# - Ollama
# - vLLM
# - LM Studio
# - Transformers

In [ ]:
# ============================================================
# Local LLM via Ollama (OpenAI-compatible endpoint)
# ============================================================

local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=300
)

local_response = local_client.responses.create(
    model="gpt-oss:20b-cloud",
    input=[
        {
            "role": "system",
            "content": (
                "You are an expert in Machine Learning "
                "and Explainable AI."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_output_tokens=12288
)

local_explanation = ""

for item in local_response.output:
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                local_explanation += content.text

print("\n===== LOCAL LLM EXPLANATION =====\n")
print(local_explanation)



===== LOCAL LLM EXPLANATION =====

**1.  Top‑3 most influential features for the intrusion‑detection model**

| Rank | Feature | Why it matters (in plain terms) |
|------|---------|---------------------------------|
| 1 | **IP reputation score** | The model “looks” at how ‘clean’ or ‘troublesome’ the source IP is. A low score (e.g., 0.1 or 0.2) is often associated with suspicious traffic because many known bad IPs are flagged by external lists. Higher scores tend to be linked with benign activity. |
| 2 | **Login attempts & failed logins** | The model treats sessions that repeatedly try to log in—especially with a backlog of failures—as potentially malicious. Attack traffic usually tries many credentials (brute‑force), producing a spike in failed logins compared to a normal, single‑try session. |
| 3 | **Unusual‑time access** | Logins that happen at night or during business‑hours outside the user’s routine are a red flag. The model gives a high weight to “unusual_time_access = 1” beca